In [ ]:
# enhanced_dgm.py
import os
import shutil
import random
import uuid
import importlib.util
import subprocess
import json
import time
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass
from abc import ABC, abstractmethod

# === CONFIG ===
@dataclass
class Config:
    """Configuration for the DGM system"""
    archive_dir: str = "archive"
    max_archive_size: int = 100
    mutation_rate: float = 0.7
    crossover_rate: float = 0.3
    selection_pressure: float = 0.8
    diversity_weight: float = 0.3
    timeout_seconds: int = 5
    log_file: str = "dgm_log.json"

DEFAULT_AGENT_TEMPLATE = """
def solve(input_data=None):
    \"\"\"
    Main solving function that should be implemented by agents.
    Args:
        input_data: Optional input data for the problem
    Returns:
        Solution result
    \"\"\"
    # Default implementation: sum of range
    return sum(range(10))
"""

# Enhanced benchmark system
BENCHMARKS = [
    {
        "name": "sum_0_to_9",
        "input": None,
        "expected": 45,
        "description": "Sum of integers from 0 to 9",
        "weight": 1.0
    },
    {
        "name": "sum_0_to_4",
        "input": None,
        "expected": 10,
        "description": "Sum of integers from 0 to 4",
        "weight": 1.0
    },
    {
        "name": "fibonacci_10",
        "input": 10,
        "expected": 55,
        "description": "10th Fibonacci number",
        "weight": 2.0
    }
]

# === MUTATION STRATEGIES ===
class MutationStrategy(ABC):
    """Abstract base class for mutation strategies"""

    @abstractmethod
    def mutate(self, code: str) -> str:
        pass

class RangeMutation(MutationStrategy):
    """Mutates range() function calls"""

    def mutate(self, code: str) -> str:
        import re
        # Find all range() calls and randomly modify them
        def replace_range(match):
            current = int(match.group(1)) if match.group(1).isdigit() else 10
            # Vary the range by ±5
            new_val = max(1, current + random.randint(-5, 5))
            return f"range({new_val})"

        return re.sub(r'range\((\d+)\)', replace_range, code)

class OperatorMutation(MutationStrategy):
    """Mutates mathematical operators"""

    def mutate(self, code: str) -> str:
        operators = {'+': '-', '-': '+', '*': '//', '//': '*'}
        for old, new in operators.items():
            if old in code and random.random() < 0.3:
                code = code.replace(old, new, 1)  # Replace only first occurrence
                break
        return code

class LogicMutation(MutationStrategy):
    """Adds or modifies logic structures"""

    def mutate(self, code: str) -> str:
        mutations = [
            self._add_conditional,
            self._add_loop_modification,
            self._add_list_comprehension
        ]
        return random.choice(mutations)(code)

    def _add_conditional(self, code: str) -> str:
        if "if " not in code and random.random() < 0.5:
            # Add a simple conditional
            lines = code.split('\n')
            for i, line in enumerate(lines):
                if 'return' in line:
                    condition = f"    if input_data and input_data > 5:\n        return {random.randint(20, 100)}\n"
                    lines.insert(i, condition)
                    break
            return '\n'.join(lines)
        return code

    def _add_loop_modification(self, code: str) -> str:
        if "for " in code:
            return code.replace("sum(range(", f"sum(x*{random.randint(1,3)} for x in range(")
        return code

    def _add_list_comprehension(self, code: str) -> str:
        if "sum(range(" in code and random.random() < 0.4:
            return code.replace("sum(range(", "sum([x for x in range(")
        return code

# === ENHANCED AGENT ===
class CodingAgent:
    """Enhanced coding agent with better mutation and tracking"""

    def __init__(self, code: str, agent_id: Optional[str] = None, generation: int = 0, parent_id: Optional[str] = None):
        self.code = code
        self.id = agent_id or str(uuid.uuid4())[:8]  # Shorter IDs
        self.generation = generation
        self.parent_id = parent_id
        self.path = Path(Config.archive_dir) / f"agent_{self.id}.py"
        self.fitness_history = []
        self.creation_time = time.time()
        self.mutation_strategies = [RangeMutation(), OperatorMutation(), LogicMutation()]
        self.write_code()

    def write_code(self):
        """Write agent code to file with metadata"""
        header = f"""# Agent ID: {self.id}
# Generation: {self.generation}
# Parent: {self.parent_id}
# Created: {time.ctime(self.creation_time)}

"""
        self.path.write_text(header + self.code)

    def run(self, input_data=None, timeout: int = 5):
        """Run agent code with timeout protection"""
        try:
            spec = importlib.util.spec_from_file_location("agent", self.path)
            agent_mod = importlib.util.module_from_spec(spec)

            # Execute with timeout (simplified - in production use proper timeout)
            spec.loader.exec_module(agent_mod)

            if hasattr(agent_mod, 'solve'):
                return agent_mod.solve(input_data)
            else:
                raise AttributeError("Agent missing solve() function")

        except Exception as e:
            raise RuntimeError(f"Agent execution failed: {str(e)}")

    def mutate(self, config: Config) -> 'CodingAgent':
        """Apply multiple mutation strategies"""
        mutated_code = self.code

        # Apply random mutations
        num_mutations = random.randint(1, 3)
        for _ in range(num_mutations):
            if random.random() < config.mutation_rate:
                strategy = random.choice(self.mutation_strategies)
                mutated_code = strategy.mutate(mutated_code)

        return CodingAgent(
            code=mutated_code,
            generation=self.generation + 1,
            parent_id=self.id
        )

    def crossover(self, other: 'CodingAgent') -> 'CodingAgent':
        """Create offspring by combining code from two parents"""
        # Simple line-based crossover
        lines1 = self.code.split('\n')
        lines2 = other.code.split('\n')

        # Take random lines from each parent
        offspring_lines = []
        for i in range(max(len(lines1), len(lines2))):
            if i < len(lines1) and i < len(lines2):
                line = random.choice([lines1[i], lines2[i]])
            elif i < len(lines1):
                line = lines1[i]
            else:
                line = lines2[i]
            offspring_lines.append(line)

        return CodingAgent(
            code='\n'.join(offspring_lines),
            generation=max(self.generation, other.generation) + 1,
            parent_id=f"{self.id}+{other.id}"
        )

    def get_complexity_score(self) -> float:
        """Calculate code complexity for diversity measurement"""
        lines = len([l for l in self.code.split('\n') if l.strip()])
        keywords = len([w for w in ['if', 'for', 'while', 'def', 'class'] if w in self.code])
        return lines + keywords * 2

# === ENHANCED EVALUATOR ===
class Evaluator:
    """Enhanced evaluation system with multiple metrics"""

    def __init__(self, benchmarks: List[Dict], config: Config):
        self.benchmarks = benchmarks
        self.config = config

    def evaluate(self, agent: CodingAgent) -> Dict[str, float]:
        """Comprehensive agent evaluation"""
        results = {
            'fitness': 0.0,
            'accuracy': 0.0,
            'diversity': 0.0,
            'robustness': 0.0,
            'benchmark_results': {}
        }

        total_weight = sum(b['weight'] for b in self.benchmarks)
        weighted_score = 0.0

        try:
            for benchmark in self.benchmarks:
                try:
                    result = agent.run(benchmark.get('input'), self.config.timeout_seconds)
                    is_correct = result == benchmark['expected']
                    score = benchmark['weight'] if is_correct else 0
                    weighted_score += score

                    results['benchmark_results'][benchmark['name']] = {
                        'result': result,
                        'expected': benchmark['expected'],
                        'correct': is_correct,
                        'score': score
                    }
                except Exception as e:
                    results['benchmark_results'][benchmark['name']] = {
                        'error': str(e),
                        'correct': False,
                        'score': 0
                    }

            results['accuracy'] = weighted_score / total_weight if total_weight > 0 else 0
            results['diversity'] = agent.get_complexity_score() / 100.0  # Normalize
            results['robustness'] = 1.0 - (len([r for r in results['benchmark_results'].values() if 'error' in r]) / len(self.benchmarks))

            # Combined fitness score
            results['fitness'] = (
                0.6 * results['accuracy'] +
                0.2 * results['diversity'] +
                0.2 * results['robustness']
            )

        except Exception as e:
            print(f"[Error] Agent {agent.id} evaluation failed: {e}")

        return results

# === ENHANCED ARCHIVE ===
class ArchiveManager:
    """Advanced archive with diversity maintenance and analytics"""

    def __init__(self, config: Config):
        self.config = config
        self.agents = []
        self.generation_stats = {}
        Path(config.archive_dir).mkdir(exist_ok=True)
        self.log_file = Path(config.log_file)

    def add(self, agent: CodingAgent, evaluation: Dict[str, float]):
        """Add agent with diversity-based selection"""
        entry = {
            'agent': agent,
            'evaluation': evaluation,
            'timestamp': time.time()
        }

        self.agents.append(entry)

        # Maintain archive size with diversity selection
        if len(self.agents) > self.config.max_archive_size:
            self._maintain_diversity()

        self._update_stats(agent.generation, evaluation)
        self._log_entry(entry)

    def _maintain_diversity(self):
        """Keep diverse, high-performing agents"""
        # Sort by fitness, keep top performers
        self.agents.sort(key=lambda x: x['evaluation']['fitness'], reverse=True)

        # Keep top 50% by fitness
        top_half = self.agents[:len(self.agents)//2]

        # Fill remaining slots with diverse agents
        remaining = self.agents[len(self.agents)//2:]
        remaining.sort(key=lambda x: x['evaluation']['diversity'], reverse=True)

        diverse_agents = []
        for agent_entry in remaining:
            if len(diverse_agents) < self.config.max_archive_size - len(top_half):
                # Check if sufficiently different from existing agents
                if self._is_sufficiently_diverse(agent_entry, top_half + diverse_agents):
                    diverse_agents.append(agent_entry)

        self.agents = top_half + diverse_agents

    def _is_sufficiently_diverse(self, candidate, existing, threshold=0.3):
        """Check if candidate is sufficiently different from existing agents"""
        if not existing:
            return True

        candidate_complexity = candidate['agent'].get_complexity_score()
        for entry in existing:
            existing_complexity = entry['agent'].get_complexity_score()
            if abs(candidate_complexity - existing_complexity) < threshold:
                return False
        return True

    def _update_stats(self, generation: int, evaluation: Dict[str, float]):
        """Update generation statistics"""
        if generation not in self.generation_stats:
            self.generation_stats[generation] = {
                'count': 0,
                'avg_fitness': 0.0,
                'max_fitness': 0.0,
                'avg_diversity': 0.0
            }

        stats = self.generation_stats[generation]
        stats['count'] += 1
        stats['avg_fitness'] = (stats['avg_fitness'] * (stats['count'] - 1) + evaluation['fitness']) / stats['count']
        stats['max_fitness'] = max(stats['max_fitness'], evaluation['fitness'])
        stats['avg_diversity'] = (stats['avg_diversity'] * (stats['count'] - 1) + evaluation['diversity']) / stats['count']

    def _log_entry(self, entry):
        """Log entry to file"""
        log_entry = {
            'timestamp': entry['timestamp'],
            'agent_id': entry['agent'].id,
            'generation': entry['agent'].generation,
            'parent_id': entry['agent'].parent_id,
            'evaluation': entry['evaluation']
        }

        # Append to log file
        logs = []
        if self.log_file.exists():
            try:
                logs = json.loads(self.log_file.read_text())
            except:
                logs = []

        logs.append(log_entry)
        self.log_file.write_text(json.dumps(logs, indent=2))

    def get_best_agents(self, n: int = 5) -> List[CodingAgent]:
        """Get top N performing agents"""
        sorted_agents = sorted(self.agents, key=lambda x: x['evaluation']['fitness'], reverse=True)
        return [entry['agent'] for entry in sorted_agents[:n]]

    def get_diverse_agents(self, n: int = 5) -> List[CodingAgent]:
        """Get most diverse agents"""
        sorted_agents = sorted(self.agents, key=lambda x: x['evaluation']['diversity'], reverse=True)
        return [entry['agent'] for entry in sorted_agents[:n]]

    def summary(self) -> Dict[str, Any]:
        """Comprehensive archive summary"""
        if not self.agents:
            return {"message": "No agents in archive"}

        evaluations = [entry['evaluation'] for entry in self.agents]

        return {
            'total_agents': len(self.agents),
            'avg_fitness': sum(e['fitness'] for e in evaluations) / len(evaluations),
            'max_fitness': max(e['fitness'] for e in evaluations),
            'avg_diversity': sum(e['diversity'] for e in evaluations) / len(evaluations),
            'generation_stats': self.generation_stats,
            'top_agents': [(entry['agent'].id, entry['evaluation']['fitness'])
                          for entry in sorted(self.agents, key=lambda x: x['evaluation']['fitness'], reverse=True)[:10]]
        }

# === ENHANCED MAIN LOOP ===
def run_enhanced_dgm(iterations: int = 50, config: Optional[Config] = None):
    """Run enhanced DGM with better evolution strategies"""
    if config is None:
        config = Config()

    archive = ArchiveManager(config)
    evaluator = Evaluator(BENCHMARKS, config)

    print("🚀 Starting Enhanced DGM System")
    print(f"📊 Benchmarks: {len(BENCHMARKS)}")
    print(f"🔄 Iterations: {iterations}")
    print(f"📁 Archive: {config.archive_dir}")

    # Initialize with first agent
    initial_agent = CodingAgent(DEFAULT_AGENT_TEMPLATE)
    evaluation = evaluator.evaluate(initial_agent)
    archive.add(initial_agent, evaluation)

    print(f"\n✅ Initial agent {initial_agent.id} - Fitness: {evaluation['fitness']:.3f}")

    for iteration in range(iterations):
        print(f"\n🔄 Iteration {iteration + 1}/{iterations}")

        # Selection: choose parents
        best_agents = archive.get_best_agents(5)
        diverse_agents = archive.get_diverse_agents(3)
        parent_pool = best_agents + diverse_agents

        if len(parent_pool) < 2:
            parent_pool = archive.get_best_agents(2) or [initial_agent, initial_agent]

        # Generate offspring
        offspring = []

        # Mutation-based offspring
        for _ in range(3):
            parent = random.choice(parent_pool)
            child = parent.mutate(config)
            offspring.append(child)

        # Crossover-based offspring
        if len(parent_pool) >= 2 and random.random() < config.crossover_rate:
            parent1, parent2 = random.sample(parent_pool, 2)
            child = parent1.crossover(parent2)
            offspring.append(child)

        # Evaluate and add offspring
        for child in offspring:
            evaluation = evaluator.evaluate(child)
            archive.add(child, evaluation)

            print(f"  👶 Agent {child.id} (Gen {child.generation}) - "
                  f"Fitness: {evaluation['fitness']:.3f}, "
                  f"Accuracy: {evaluation['accuracy']:.3f}, "
                  f"Diversity: {evaluation['diversity']:.3f}")

        # Progress update every 10 iterations
        if (iteration + 1) % 10 == 0:
            stats = archive.summary()
            print(f"\n📈 Progress Update - Iteration {iteration + 1}")
            print(f"   Archive Size: {stats['total_agents']}")
            print(f"   Best Fitness: {stats['max_fitness']:.3f}")
            print(f"   Avg Fitness: {stats['avg_fitness']:.3f}")

    # Final results
    print("\n" + "="*60)
    print("🏆 FINAL RESULTS")
    print("="*60)

    summary = archive.summary()
    print(f"📊 Archive Summary:")
    print(f"   Total Agents: {summary['total_agents']}")
    print(f"   Best Fitness: {summary['max_fitness']:.3f}")
    print(f"   Average Fitness: {summary['avg_fitness']:.3f}")
    print(f"   Average Diversity: {summary['avg_diversity']:.3f}")

    print(f"\n🥇 Top 5 Agents:")
    for i, (agent_id, fitness) in enumerate(summary['top_agents'][:5], 1):
        print(f"   {i}. Agent {agent_id}: {fitness:.3f}")

    # Show best agent code
    best_agents = archive.get_best_agents(1)
    if best_agents:
        best_agent = best_agents[0]
        print(f"\n🏆 Best Agent Code ({best_agent.id}):")
        print("-" * 40)
        print(best_agent.code)
        print("-" * 40)

    print(f"\n📁 Results saved to: {config.archive_dir}")
    print(f"📋 Logs saved to: {config.log_file}")

    return archive

def main(iterations=30, archive_size=50, mutation_rate=0.7, archive_dir="archive"):
    """Main entry point with configuration options"""
    config = Config(
        archive_dir=archive_dir,
        max_archive_size=archive_size,
        mutation_rate=mutation_rate
    )

    try:
        archive = run_enhanced_dgm(iterations, config)
        print("\n✅ DGM completed successfully!")
        return archive
    except KeyboardInterrupt:
        print("\n⚠️  Process interrupted by user")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        raise

def main_cli():
    """Command line interface version"""
    import argparse
    import sys

    # Check if we're in a notebook environment
    try:
        get_ipython()
        # We're in IPython/Jupyter, don't use argparse
        print("🔬 Detected notebook environment - using default parameters")
        return main()
    except NameError:
        # We're in regular Python, use argparse
        pass

    parser = argparse.ArgumentParser(description="Enhanced Diverse Generation and Maintenance System")
    parser.add_argument("--iterations", type=int, default=30, help="Number of iterations")
    parser.add_argument("--archive-size", type=int, default=50, help="Maximum archive size")
    parser.add_argument("--mutation-rate", type=float, default=0.7, help="Mutation rate")
    parser.add_argument("--archive-dir", type=str, default="archive", help="Archive directory")

    # Filter out Jupyter-specific arguments
    filtered_args = [arg for arg in sys.argv[1:] if not arg.startswith('-f')]
    args = parser.parse_args(filtered_args)

    return main(
        iterations=args.iterations,
        archive_size=args.archive_size,
        mutation_rate=args.mutation_rate,
        archive_dir=args.archive_dir
    )

if __name__ == "__main__":
    main_cli()

🔬 Detected notebook environment - using default parameters
🚀 Starting Enhanced DGM System
📊 Benchmarks: 3
🔄 Iterations: 30
📁 Archive: archive

✅ Initial agent cfe15bdb - Fitness: 0.378

🔄 Iteration 1/30
  👶 Agent f9571556 (Gen 1) - Fitness: 0.228, Accuracy: 0.000, Diversity: 0.140
  👶 Agent 3035c01e (Gen 1) - Fitness: 0.228, Accuracy: 0.000, Diversity: 0.140
  👶 Agent 9b7dba5a (Gen 1) - Fitness: 0.228, Accuracy: 0.000, Diversity: 0.140

🔄 Iteration 2/30
  👶 Agent c08fab1d (Gen 2) - Fitness: 0.228, Accuracy: 0.000, Diversity: 0.140
  👶 Agent 9a3966e6 (Gen 1) - Fitness: 0.378, Accuracy: 0.250, Diversity: 0.140
  👶 Agent 09eba8d7 (Gen 1) - Fitness: 0.028, Accuracy: 0.000, Diversity: 0.140

🔄 Iteration 3/30
  👶 Agent 87be4de5 (Gen 2) - Fitness: 0.228, Accuracy: 0.000, Diversity: 0.140
  👶 Agent aa9b7d15 (Gen 1) - Fitness: 0.386, Accuracy: 0.250, Diversity: 0.180
  👶 Agent f5754538 (Gen 2) - Fitness: 0.378, Accuracy: 0.250, Diversity: 0.140

🔄 Iteration 4/30
  👶 Agent 3ac25474 (Gen 2) - Fit